# Đánh giá DG · B2 + DeepLab · sáu best checkpoint

**Chỉ đánh giá, không train.** Một lần Run All chạy 6 checkpoint × (Cityscapes val500 + ACDC val406). Dùng FP32, canvas512×1024 giữ tỷ lệ, output native, ignore255, không TTA.

**Thêm đúng 4 input:** ZIP B2 hoàn chỉnh `(1)`, ZIP DeepLab hoàn chỉnh `(1)`, Cityscapes, ACDC. ZIP hoặc dataset kết quả đã giải nén đều được; không cần code ZIP riêng. Không dùng Release DeepLab epoch4, ZIP B0 hoặc kết quả supervised cũ.

Tạo notebook mới, bật GPU/Internet, giữ Private. Chỉ sửa bốn đường dẫn bên dưới nếu auto-find báo thiếu hoặc nhiều kết quả. Source và sáu checkpoint được đối chiếu hash đã nghiệm thu trước khi đọc ảnh target. Không chọn lại best/tune theo ACDC. ACDC đã được xem trong phát triển lịch sử, chưa phải test độc lập mới.

Giữ ZIP sau mỗi phiên. Resume evaluation chỉ bỏ qua job đã hoàn thành và kiểm hash; job đang dở chạy lại từ đầu. Hard timeout có thể bỏ qua finally; ZIP được cập nhật sau mỗi job hoàn chỉnh. Notebook chưa được chạy đủ 12 job trên Kaggle tại lúc bàn giao.

In [ ]:
from pathlib import Path, PurePosixPath
from contextlib import contextmanager
import csv, hashlib, io, json, os, shutil, stat, subprocess, sys, time, zipfile

ARMS = ('baseline', 'aug', 'consistency')
MODELS = ('segformer_b2', 'deeplabv3plus_resnet50')

def sha(path):
    with Path(path).open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

def safe_name(name):
    p = PurePosixPath(name.replace(chr(92), '/'))
    if p.is_absolute() or '..' in p.parts or ':' in str(p):
        raise ValueError('Unsafe archive path: ' + name)
    return str(p)

@contextmanager
def reader(path):
    path = Path(path)
    if path.is_dir():
        yield lambda name: (path/safe_name(name)).read_bytes()
    else:
        with zipfile.ZipFile(path) as z:
            names = [safe_name(i.filename).rstrip('/').casefold() for i in z.infolist()]
            if len(names) != len(set(names)):
                raise ValueError('Duplicate archive paths')
            if any(stat.S_IFMT(i.external_attr >> 16) == stat.S_IFLNK for i in z.infolist()):
                raise ValueError('Archive symlinks are not supported')
            yield z.read

def suite_identity(path, model):
    try:
        with reader(path) as read:
            plan = json.loads(read('suite_plan.json'))
            status = json.loads(read('suite_status.json'))
        return (plan.get('schema') == 'automatic_dg_model_suite_v1'
            and plan.get('protocol') == 'cityscapes_only_dg_v1'
            and plan.get('target_data_access') == 'none'
            and plan.get('model') == model and plan.get('seed') == 42
            and set(plan.get('arms', [])) == set(ARMS)
            and status.get('status') == 'suite_complete' and status.get('completed_arms') == 3)
    except (OSError, KeyError, ValueError, zipfile.BadZipFile):
        return False

def find_suite(hint, model, input_root):
    base = Path(hint) if hint is not None else Path(input_root)
    if suite_identity(base, model):
        return base
    if not base.is_dir():
        raise ValueError(f'{model}: missing or incomplete suite: {base}')
    folders = sorted({p.parent for p in base.rglob('suite_plan.json') if suite_identity(p.parent, model)})
    found = folders or [p for p in sorted(base.rglob('*.zip')) if suite_identity(p, model)]
    if len(found) != 1:
        raise ValueError(f'{model}: expected exactly one COMPLETE suite, found {found}. Set its input path explicitly.')
    return found[0]

def stage_suite(path, model, destination, expected_source):
    destination = Path(destination)
    with reader(path) as read:
        plan = json.loads(read('suite_plan.json'))
        if plan['source_hashes'] != expected_source:
            raise ValueError('Training source differs from the audited cohort')
        # Copy only metadata and initial/best weights. No optimizer/last weights,
        # dataset masks, source snapshots per arm, or full8GB duplicate.
        names = ['suite_plan.json', 'suite_status.json']
        for arm in ARMS:
            prefix = f'jobs/{arm}/'
            job = json.loads(read(prefix+'dg_plan.json'))
            names += [prefix+n for n in set(job['frozen_files']) | {
                'frozen_candidate.json', 'dg_plan.json', 'run/config.json',
                'run/history.jsonl', 'run/resources.jsonl', 'run/best/checkpoint.json',
                'run/last/checkpoint.json', 'run/best/config.json',
                'run/best/model.safetensors', 'run/initial/model.safetensors'}]
        for name in sorted(set(names)):
            target = destination/safe_name(name)
            data = read(name)
            if target.is_file():
                if hashlib.sha256(data).hexdigest() != sha(target):
                    raise ValueError('Preserve changed staged file: ' + str(target))
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(data)
    return destination

def restore_source(suite, destination, expected):
    destination = Path(destination)
    with reader(suite) as read:
        try:
            raw = read('source.zip')
        except (FileNotFoundError, KeyError):
            raw = None
        if raw is not None:
            with zipfile.ZipFile(io.BytesIO(raw)) as z:
                contents = {name:z.read(name) for name in expected}
        else:
            # Kaggle can expand a nested source.zip into source/.
            contents = {name:read('source/'+name) for name in expected}
    if any(hashlib.sha256(contents[n]).hexdigest() != h for n,h in expected.items()):
        raise ValueError('Frozen source hash mismatch; nothing will be imported')
    for name,data in contents.items():
        target = destination/safe_name(name)
        if target.exists() and sha(target) != expected[name]:
            raise ValueError('Preserve changed evaluation source: ' + name)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
    return destination

def find_tree(hint, input_root, anchor, required):
    base = Path(hint) if hint is not None else Path(input_root)
    good = lambda p: all((p/n).is_dir() for n in required)
    if good(base):
        return base
    if not base.is_dir():
        raise ValueError('Missing dataset path: ' + str(base))
    found = sorted({p.parent for p in base.rglob(anchor) if p.is_dir() and good(p.parent)})
    if len(found) != 1:
        raise ValueError(f'Expected one {anchor} dataset tree, found {found}; set its path explicitly')
    return found[0]

def completed_evaluation(path, expected_weights, manifest, counts):
    path = Path(path)
    if not path.exists():
        return False
    state_file, metrics_file = path/'run-state.json', path/'metrics.json'
    if not state_file.is_file() or not metrics_file.is_file():
        return False
    state = json.loads(state_file.read_text())
    if state.get('status') != 'complete':
        return False
    if (state['checkpoint_sha256'] != expected_weights or state['manifest_sha256'] != sha(manifest)
        or state['metrics_sha256'] != sha(metrics_file) or state['image_size'] != [512,1024]
        or state['dtype'] != 'float32' or state['tta'] is not False):
        raise ValueError('Cached evaluation identity changed; preserve it and use a new output folder')
    metrics = json.loads(metrics_file.read_text())
    if metrics['n_images'] != sum(counts.values()) or {d:v['n_images'] for d,v in metrics['by_domain'].items()} != counts:
        raise ValueError('Cached evaluation has incorrect split counts')
    if len(state['cache_sha256']) != sum(counts.values()):
        raise ValueError('Incomplete confusion-matrix cache')
    for name,h in state['cache_sha256'].items():
        if sha(path/safe_name(name)) != h:
            raise ValueError('Cached confusion matrix changed')
    return True

EXPECTED_SOURCE = {'src/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'src/data/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'src/data/acdc.py': '700fe82b258ffbc5728a4cc78e6eedf9f666b9330c19a6f539580beed9f932aa', 'src/data/augmentations/__init__.py': '2c7610ca422d29a4bd0f756ccc574f12d628d9b08bb4be41bc68efdaba3e3e41', 'src/data/augmentations/class_aware_crop.py': 'e9ff34505264f7fe7b2e63c7802f69a2db2fe9cdfcab3b964943ead8de6dfa22', 'src/data/augmentations/gamma.py': 'd0b0592c59b9a6f129f75f097f3baf0fc7b71170583085e185ca85bff7cf130e', 'src/data/augmentations/photometric.py': 'f81197c25dec5f2a2c00f3914bf885500991962d550fb71d5166f47e9f787d94', 'src/data/augmentations/source_style.py': 'e7b9803c510b2afc1d9d0f0072985719717ee5cce6a7b9b2d3b57380c13a5b61', 'src/data/class_weights.py': 'f6cf72af3040c405363a1999ff460fd9b00eba09797a98d3633c41060f25c5f5', 'src/data/dataset.py': '3aa6e3533d5c96076c4d304e9e8cb873e5c0affb646cfde208a389d873b3427d', 'src/data/labels.py': 'aaaeb9b4ff79d34a91657b4f84eda329a5e9f2137824de46540f9a98f702e26e', 'src/data/partitions.py': 'df910c7a4649e95c961348a7545de6432f071cc0f1dafd1ecc54c9e99f3dc530', 'src/data/preparation.py': '6fcc33babae1cf9e3086ac5aaa20121b7c97e6e2edb0c50695bab3c6749bc94a', 'src/data/preprocessing.py': 'a30c473000341590ec6771fca7b24bb72d11c7b9838b0bfe39742c101f21167e', 'src/data/replay.py': 'b88ba6063afcb3512eac7ec698659a474af913db8c6f9ddc155ad2ff6b10e238', 'src/data/sampling.py': 'dc5fc3828f7cc9b2fd2bc1132226c2f084cb5e9440b1c5cc48d062fc49333589', 'src/data/validation.py': 'cae5f5c6e9ce2bb00a53978f84b9a1de6f9f42ace57322e805861df3203b54f0', 'src/evaluation/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'src/evaluation/calibration.py': '3e2b66ecfc58f71110c01d8010c3bd9474b5c90162c14b0112152b7c788b8608', 'src/evaluation/metrics.py': 'cd071faaf74853ebf6288c5a3a1548135ac383ebc00552e156fe5e06997a9d75', 'src/evaluation/runner.py': '2ee8bf705857421f1be83130ef0866f54307b27e91e03d1360480bba5687ed07', 'src/evaluation/semantic.py': 'c29d6283021a53060908c5f32290cf65841b2c5e63340ee4514b3f5c7623739e', 'src/experiments/__init__.py': 'aea6002fff8757d3392d4eb11346b5d652b3b7f9779d12e47d6abc9322c90d14', 'src/experiments/config.py': '6e9c814dd273b4629b2c55e559db924c27a38de901cf0281e8705f380bebd3a7', 'src/experiments/dg.py': '847f2dfce6922345b1a59513e42880614b318bc284062012ccd2492988391e85', 'src/experiments/dg_artifacts.py': '7fa06c9757ce71f418b2ce70bb3c0d9f4a9f70a7e13fcab9b52b64dbc699474b', 'src/experiments/packaging.py': '6ae4637b16f08618273f617cbfe2a084a5ef52261a1e635a943f3b4a81d06420', 'src/experiments/protocol.py': '238a7e027923e897063ba5493894d64d8d96c61b94280b413bad2254e125b722', 'src/experiments/run_support.py': '6f07207c3393bd2cd5ded90070edd4ce82205752641abff1d2cbc3c57162637f', 'src/inference.py': '75864f6d1e1ded39a5748e91c08334a40cb24b654a6a5d75ef4615610d83e775', 'src/models/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'src/models/deeplabv3plus_resnet50.py': 'f89b49b925367a82bbbf4e6513ebc95bf79788fd4171959da71730356c8825c6', 'src/models/legacy/__init__.py': '2958231a6f6ab31c6216341451692a73a41c52e2cd980ab0d4f319b472425c4b', 'src/models/legacy/segformer_b0.py': '1c4218faab68ce128101964a5bc39cfa0180eba1bab2c6144e414b1d69a21b83', 'src/models/registry.py': 'a5a3f7e70549af0cda050fc1c6c7af8ff774a879d5c4e2f71941388396b0e183', 'src/models/segformer.py': 'e09e27fed32dee129969d1cc7d114db3272c014d8db004f12a071c76156baed2', 'src/models/segformer_b2.py': '0b5512399ce9450ad7d5a2b711fb734df8d13aa52cf614c1ad13143f66c9284f', 'src/training/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'src/training/checkpoints.py': 'd6524e7d8a34e73568f7c2b17da31e23445a703622be0c6dbc1b9f6ea870f361', 'src/training/dg.py': 'b45d5f2e4bf63e5d0a1d515810da9a9c8d2049da21c89aac782db85832b5b5e3', 'src/training/losses.py': '8cfe56a271d718c398e94905106e5907905dae78097331f3f5f2a0487385132a', 'src/training/optim.py': 'db30ce5c1db7dee8386fe75983a96ca833caa22a61d500288ba6b0fc104af9b1', 'src/training/strategies.py': '7a76c41d446dfd4531dfa867008c37a8b3fd7c9ff670066b868903562f7e4877', 'src/training/train.py': 'c4a954037b4c74b9995c82afe8ed73ff16b212b6a5f55ec4df2a11b1154aea59', 'src/utils/__init__.py': 'c0ace31fc4c6b9d999d9002fa5250fb8950e25377e1cc3218b954b8ca222901e', 'src/utils/artifacts.py': '100f16261204cb205b7eb6fb66dceef0fe81a16c903eb35ece7cb95a2504267d', 'scripts/audit_labels.py': '8acd170b01d0b2926433e04e380bbcc0a473b1f96ca8f48b7bcb2bb9e38cd3d5', 'scripts/build_dg_notebook.py': '6cdc940a604ab963ee60e6b84701b2dbcda62490af097c0b55e3c9b67e8529d2', 'scripts/build_kaggle_notebooks.py': '937e1caaf1430248467708e3a8c3b41cf7e7d08b8f63d8c8fe8344e4cc2801f6', 'scripts/convert_labels.py': '9fbc4792d9ca1c1ed74e3be30162b668d27871a0710eab817ece4730d52e7f93', 'scripts/evaluate_dg.py': 'fc1453537fae72fd5cbd02e18c2419759d25d585522de0467ea907a938a84844', 'scripts/extract_dataset.py': '1f9471016b0f779783474826ebcbc68e9d42eb038b40cd701b832d366e8b2cfa', 'scripts/kaggle_bootstrap.py': '521aa5d63fa4f12a5b1365692ca0972fc31f20bf8bb77e40fd306114dea00b6c', 'scripts/make_manifest.py': '587329de874b9dae5b163d239961ced299a80b40f823a9f44fa5318e2c926f06', 'scripts/plot_research_run.py': '1df8eef0e2ba7b95fe1494c8bce9416e468c043536f8ee802d057bea3c6f323a', 'scripts/profile_dg.py': 'c5206d046635c3d8012dae060f99ca45d2b578230b4147ffd87c1387d9ae0fa0', 'scripts/run_dg.py': '28adc0616bf10bf5b0b71fd5ba6779716f11a2ef6084820d8b901604e494e69c', 'scripts/run_dg_suite.py': '0ffed3a8e0a662004b078f0643775c13c290abf7c6e660896a0fb2e06568bc1f', 'scripts/validate_manifest.py': '1272ffe363720fdd668d2423cbb5603732a4ee926b29cabca2057988d4877b52', 'scripts/verify_dg.py': 'ccbb490176e8238d1403328432c417b71a8aad584c78541f1e9c91a5ec88a1d9', 'configs/dg/paired_source.yaml': '354fa49fc0554bb684e9af037131e4d86be304d58799d6d9dbed95bcab08e69d', 'configs/losses/ce.yaml': 'abb504d6bbd5c02512d1a74c7e09c9bf28ed575505ceff39c6f4582481b8b8e0', 'configs/losses/ce_dice.yaml': '8f92c21fb27f75c7eb0a601b22abff8383e31f1a7d0abe349d46ad7ec7f24ae8', 'configs/losses/focal.yaml': '99579039844a411c503d47649e0cd024236dc374abc447052e62ceaa65a4aa7a', 'configs/losses/weighted_ce.yaml': '47e12a2db7855c26649af4b11c368bd0f26f4ce2496c427ba1dba115082fc662', 'configs/preprocessing/shared_512.yaml': '0b76f47f6b150941cf857fe70ba0f607f3b8432d69b83ff7b858d34fa5677fa9', 'configs/recipes/adaptation_20epoch.yaml': 'e7e5a2715aaf8c356ab2455da03538ad88877a2ece7b25fe099da3d8f195d57c', 'configs/recipes/source_20epoch.yaml': '1927e613aa553bc0914e5a9643e3523cc1a48a8f81b7a3b61c389c401a6ea271'}
EXPECTED_CANDIDATES = [{'id': 'segformer_b2:baseline', 'candidate_sha256': 'fefc14610873b6991a769aaf1b9ef853e2efdcc8ab6982cf39d8b7740e41d7c5', 'weights_sha256': '2179af05a6cbc898f1bada5a379d11ea2cdb2b16faba826321e2bfb6263fd05b', 'initial_sha256': 'f9b5af554a725a0864e5dd8788a569655d6faab7625013f3f32ea0efb958406a', 'source_ids': {'train': '9e1b2d111b2a0966ea765df7a1d792058cc8e57c827d62791d9f1d702962f26c', 'val': 'cbc9fc372d49138ad85143ef5a93ad7d506345dcf173c49874027cbd848a8ee8'}, 'seed': 42, 'model': 'segformer_b2', 'arm': 'baseline'}, {'id': 'segformer_b2:aug', 'candidate_sha256': '9468a74618adfb81a35b7ba77cb756357ed783fa143cc8b480427d356984ac6a', 'weights_sha256': '1e4cad94d9983038fe9e4a591b85030639b90cba6313c14c63ac41bc99b172cf', 'initial_sha256': 'f9b5af554a725a0864e5dd8788a569655d6faab7625013f3f32ea0efb958406a', 'source_ids': {'train': '9e1b2d111b2a0966ea765df7a1d792058cc8e57c827d62791d9f1d702962f26c', 'val': 'cbc9fc372d49138ad85143ef5a93ad7d506345dcf173c49874027cbd848a8ee8'}, 'seed': 42, 'model': 'segformer_b2', 'arm': 'aug'}, {'id': 'segformer_b2:consistency', 'candidate_sha256': '695c106b9a4bdf325550f929cc323d5529ac3bc378812e3b1f99993354a38219', 'weights_sha256': 'f2f72b920cfe7138a916abd08dd2d293c124ecbcaca0f3ed49aa129423a5a6f2', 'initial_sha256': 'f9b5af554a725a0864e5dd8788a569655d6faab7625013f3f32ea0efb958406a', 'source_ids': {'train': '9e1b2d111b2a0966ea765df7a1d792058cc8e57c827d62791d9f1d702962f26c', 'val': 'cbc9fc372d49138ad85143ef5a93ad7d506345dcf173c49874027cbd848a8ee8'}, 'seed': 42, 'model': 'segformer_b2', 'arm': 'consistency'}, {'id': 'deeplabv3plus_resnet50:baseline', 'candidate_sha256': '60d03ecd2803e1732c813a453db09055314b4e8282aadfe1cf5d258e3603e0fa', 'weights_sha256': 'ba995063ab20b854e787c3d0e9bda09d002dbdb277e4143f137e204588e8db4b', 'initial_sha256': '66ab1aaa9fdb9051cf8b156941827c838272e0990877706fed859218944d66fd', 'source_ids': {'train': '9e1b2d111b2a0966ea765df7a1d792058cc8e57c827d62791d9f1d702962f26c', 'val': 'cbc9fc372d49138ad85143ef5a93ad7d506345dcf173c49874027cbd848a8ee8'}, 'seed': 42, 'model': 'deeplabv3plus_resnet50', 'arm': 'baseline'}, {'id': 'deeplabv3plus_resnet50:aug', 'candidate_sha256': '2c0b462093721b08fece8892e56fbf01a14379fda4e3729800fe10a30c1f8f3a', 'weights_sha256': '840c94c40c66c5f14e9d244b7352eeb6410c032cbd5357145940578e6647304f', 'initial_sha256': '66ab1aaa9fdb9051cf8b156941827c838272e0990877706fed859218944d66fd', 'source_ids': {'train': '9e1b2d111b2a0966ea765df7a1d792058cc8e57c827d62791d9f1d702962f26c', 'val': 'cbc9fc372d49138ad85143ef5a93ad7d506345dcf173c49874027cbd848a8ee8'}, 'seed': 42, 'model': 'deeplabv3plus_resnet50', 'arm': 'aug'}, {'id': 'deeplabv3plus_resnet50:consistency', 'candidate_sha256': '162b0e9c69b3a3a326be7f02869971595f7293de9f82e720cdcaaa9150a27890', 'weights_sha256': '0fc2f2abb91c492c4974af858ebf2854830a46bc3e83d682b8b83dfb07a7a1c0', 'initial_sha256': '66ab1aaa9fdb9051cf8b156941827c838272e0990877706fed859218944d66fd', 'source_ids': {'train': '9e1b2d111b2a0966ea765df7a1d792058cc8e57c827d62791d9f1d702962f26c', 'val': 'cbc9fc372d49138ad85143ef5a93ad7d506345dcf173c49874027cbd848a8ee8'}, 'seed': 42, 'model': 'deeplabv3plus_resnet50', 'arm': 'consistency'}]


In [ ]:
# These are the only four REQUIRED inputs. None = auto-find.
B2_INPUT = None          # COMPLETE dg_segformer_b2_suite_seed42 (1).zip or extracted dataset
DEEPLAB_INPUT = None     # COMPLETE dg_deeplabv3plus_resnet50_suite_seed42 (1).zip or extracted dataset
CITYSCAPES_ROOT = None   # Folder containing leftImg8bit/val and gtFine/val
ACDC_ROOT = None         # ACDC common parent; rgb_anon_trainvaltest/ and gt_trainval/ below it
RESUME_EVAL_INPUT = None # Optional: returned dg_evaluation_seed42.zip after an interrupted evaluation

INPUT = Path('/kaggle/input')
BASE = Path('/kaggle/working')
OUT = BASE/'dg_evaluation_seed42'
REPO = BASE/'project_dg_evaluation'
STAGING = BASE/'dg_evaluation_inputs'
ARCHIVE = BASE/'dg_evaluation_seed42.zip'
assert not OUT.exists() and not REPO.exists() and not STAGING.exists(), 'Use a fresh session; preserve existing outputs first'
sources = {model:find_suite(hint,model,INPUT) for model,hint in zip(MODELS,[B2_INPUT,DEEPLAB_INPUT])}
roots = {model:stage_suite(sources[model],model,STAGING/model,EXPECTED_SOURCE) for model in MODELS}
restore_source(sources['segformer_b2'],REPO,EXPECTED_SOURCE)
CITYSCAPES_ROOT = find_tree(CITYSCAPES_ROOT,INPUT,'leftImg8bit',['leftImg8bit/val','gtFine/val'])
ACDC_RGB_ROOT = find_tree(ACDC_ROOT,INPUT,'rgb_anon',[f'rgb_anon/{d}/val' for d in ['fog','night','rain','snow']])
ACDC_GT_ROOT = find_tree(ACDC_ROOT,INPUT,'gt',[f'gt/{d}/val' for d in ['fog','night','rain','snow']])
if RESUME_EVAL_INPUT is not None:
    previous = Path(RESUME_EVAL_INPUT)
    if previous.is_file():
        with reader(previous) as read, zipfile.ZipFile(previous) as z:
            for item in z.infolist():
                if not item.is_dir():
                    target = OUT/safe_name(item.filename)
                    target.parent.mkdir(parents=True,exist_ok=True); target.write_bytes(read(item.filename))
    else:
        candidates = [previous] if (previous/'evaluation_plan.json').is_file() else [p.parent for p in previous.rglob('evaluation_plan.json')]
        assert len(candidates)==1, 'Set RESUME_EVAL_INPUT to the returned evaluation result'
        shutil.copytree(candidates[0],OUT)
os.chdir(REPO);sys.path.insert(0,str(REPO))
os.environ.update(USE_TF='0',USE_TORCH='1',OMP_NUM_THREADS='2',MKL_NUM_THREADS='2')
for model,p in sources.items(): print(model,':',p)
print('Cityscapes:',CITYSCAPES_ROOT,'\nACDC RGB:',ACDC_RGB_ROOT,'\nACDC GT:',ACDC_GT_ROOT)


## 1. Cài thư viện và khóa sáu checkpoint
Không cần tải pretrained weights; dùng best đã học trong hai suite.

In [ ]:
def run(*args):
    subprocess.run([sys.executable,'-u',*map(str,args)],cwd=REPO,check=True)
run('-m','pip','install','transformers==4.57.6','numpy>=1.26,<3','Pillow==11.3.0',
    'PyYAML>=6,<7','matplotlib>=3.8,<4','safetensors>=0.4')
import torch
assert torch.cuda.is_available(), 'Enable GPU in Notebook settings'
torch.set_num_threads(2)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print('GPU:',torch.cuda.get_device_name(0),'Torch:',torch.__version__)
from src.experiments.dg_artifacts import seal_suites
cohort_path = STAGING/'cohort.json'
sealed = seal_suites(list(roots.values()),cohort_path)
expected = {c['id']:c for c in EXPECTED_CANDIDATES}
for c in sealed['candidates']:
    assert all(c[k]==expected[c['id']][k] for k in expected[c['id']]), 'Not the audited best checkpoint'
OUT.mkdir(parents=True,exist_ok=True)
plan = dict(schema='frozen_six_candidate_evaluation_v1',
    candidates=EXPECTED_CANDIDATES, source_hashes=EXPECTED_SOURCE,
    scopes=['official-city','official-acdc'], canvas=[512,1024],dtype='float32',tta=False,
    policy='No training, target BN, checkpoint selection or tuning on ACDC',
    historical_feedback_limit='ACDC was viewed during earlier non-DG development; not a pristine independent test')
plan_path = OUT/'evaluation_plan.json'
if plan_path.exists(): assert json.loads(plan_path.read_text())==plan, 'Evaluation design changed'
else: plan_path.write_text(json.dumps(plan,indent=2)+'\n')
shutil.copy2(cohort_path,OUT/'cohort.json')
print('PASS: all six audited best checkpoints frozen; no target pixels opened yet')


## 2. Kiểm input ảnh–mask và tạo manifest đánh giá
Chỉ mở val; không dùng ACDC train. Kiểm nhãn, số lượng và geometry trước inference.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from src.data.preparation import create_manifest,write_manifest,convert_label_ids
from src.data.validation import validate_manifests

assert sealed['status']=='cohort_frozen_before_target_evaluation' and len(sealed['candidates'])==6
# Build evaluation manifests at the current mount; never edit frozen train CSVs.
mask_root = STAGING/'city_masks'
city_images = sorted((CITYSCAPES_ROOT/'leftImg8bit/val').rglob('*_leftImg8bit.png'))
assert len(city_images)==500, f'Need Cityscapes val500, found {len(city_images)}'
def convert_one(image):
    name = image.name.replace('_leftImg8bit.png','_gtFine_labelTrainIds.png')
    native = CITYSCAPES_ROOT/'gtFine/val'/image.parent.name/name
    target = mask_root/'gtFine/val'/image.parent.name/name
    if native.is_file():
        target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(native,target)
    else:
        convert_label_ids(native.with_name(name.replace('_labelTrainIds.png','_labelIds.png')),target)
with ThreadPoolExecutor(max_workers=4) as pool: list(pool.map(convert_one,city_images))
manifest_dir = OUT/'manifests';manifest_dir.mkdir(exist_ok=True)
manifests = {}
for scope,rows in [
    ('official-city',create_manifest(CITYSCAPES_ROOT,'cityscapes','val',mask_root)),
    ('official-acdc',create_manifest(ACDC_RGB_ROOT,'acdc','val',ACDC_GT_ROOT))]:
    path = manifest_dir/(scope+'.csv')
    if path.exists():
        # Same mounts are required to reuse completed evaluation jobs.
        from src.data.validation import read_manifest
        assert read_manifest(path)==rows, 'Dataset mount changed; cannot reuse old evaluation cache'
    else: write_manifest(rows,path)
    manifests[scope]=path
    print(scope,':',len(rows),'images')
# The generic validator deliberately blocks target-label scans. Use its file/metadata
# check; the evaluation dataset validates label values/geometry as each image is read.
validate_manifests(list(manifests.values()),check_files=True)
# Validate real geometry and preprocessing on one image from each domain.
from src.data.dataset import ManifestSegmentationDataset
checked = set()
for scope,path in manifests.items():
    ds=ManifestSegmentationDataset(path,(512,1024),native_labels=True,allow_target_train=True)
    for i,row in enumerate(ds.items):
        if row['domain'] not in checked:
            item=ds[i];assert item['pixel_values'].shape==(3,512,1024)
            assert torch.isfinite(item['pixel_values']).all();checked.add(row['domain'])
print('PASS: files/labels and five-domain geometry smoke',sorted(checked))


## 3. Chạy tự động cả 12 phép đánh giá
Không sửa model/arm. Mỗi job hoàn chỉnh cập nhật ZIP kết quả; có panels và confusion matrix từng ảnh.

In [ ]:
from scripts.evaluate_dg import evaluate_frozen
from src.experiments.packaging import export_workspace
from IPython.display import FileLink,display

counts = {'official-city':{'cityscapes':500},'official-acdc':{'fog':100,'night':106,'rain':100,'snow':100}}
try:
    for candidate in sealed['candidates']:
        run_id=candidate['id'];slug=run_id.replace(':','__')
        for scope in plan['scopes']:
            destination=OUT/'evaluations'/slug/scope
            if completed_evaluation(destination,candidate['weights_sha256'],manifests[scope],counts[scope]):
                print('Verified completed evaluation; skip:',run_id,scope,flush=True);continue
            if destination.exists():
                preserved=OUT/'incomplete_attempts'/slug/(scope+'_'+str(time.time_ns()))
                preserved.parent.mkdir(parents=True,exist_ok=True);shutil.move(str(destination),preserved)
            print('Evaluate:',run_id,scope,flush=True)
            result=evaluate_frozen(cohort_path,run_id,manifests[scope],destination,scope,device='cuda')
            print(run_id,scope,'mIoU:',round(100*result['miou'],3),flush=True)
            assert completed_evaluation(destination,candidate['weights_sha256'],manifests[scope],counts[scope])
            export_workspace(OUT,ARCHIVE)
finally:
    export_workspace(OUT,ARCHIVE)
    print('Download:',ARCHIVE);display(FileLink(str(ARCHIVE)))


## 4. Bảng điểm, chênh lệch chiến thuật và biểu đồ
Gửi lại `dg_evaluation_seed42.zip`. mIoU gộp tính từ confusion matrix toàn tập, không lấy trung bình điểm từng ảnh hoặc trung bình bốn thời tiết.

In [ ]:
from src.data.labels import CLASS_NAMES
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
summary_rows=[];class_rows=[];weather_rows=[]
for candidate in sealed['candidates']:
    for scope in plan['scopes']:
        p=OUT/'evaluations'/candidate['id'].replace(':','__')/scope
        assert completed_evaluation(p,candidate['weights_sha256'],manifests[scope],counts[scope])
        m=json.loads((p/'metrics.json').read_text())
        summary_rows.append(dict(model=candidate['model'],arm=candidate['arm'],scope=scope,
            miou=m['miou'],parameters=m['parameters'],evaluation_seconds=m['evaluation_seconds']))
        for name,value in zip(CLASS_NAMES,m['per_class_iou']):
            class_rows.append(dict(model=candidate['model'],arm=candidate['arm'],scope=scope,class_name=name,iou=value))
        for domain,v in m['by_domain'].items():
            weather_rows.append(dict(model=candidate['model'],arm=candidate['arm'],domain=domain,miou=v['miou'],n_images=v['n_images']))
def save_csv(name,rows):
    with (OUT/name).open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
save_csv('summary.csv',summary_rows);save_csv('per_class.csv',class_rows);save_csv('per_weather.csv',weather_rows)
lookup={(r['model'],r['arm'],r['scope']):r['miou'] for r in summary_rows}
deltas=[]
for model in MODELS:
    for scope in plan['scopes']:
        deltas.append(dict(model=model,scope=scope,
            aug_minus_baseline_pp=100*(lookup[model,'aug',scope]-lookup[model,'baseline',scope]),
            consistency_minus_aug_pp=100*(lookup[model,'consistency',scope]-lookup[model,'aug',scope])))
save_csv('strategy_deltas.csv',deltas)
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,scope in zip(axes,plan['scopes']):
    for j,model in enumerate(MODELS):
        ax.bar([i+j*.36 for i in range(3)],[100*lookup[model,a,scope] for a in ARMS],width=.36,label=model)
    ax.set_xticks([i+.18 for i in range(3)],ARMS);ax.set_ylabel('mIoU (%)');ax.set_title(scope);ax.legend(fontsize=8)
fig.tight_layout();fig.savefig(OUT/'comparison.png',dpi=150);plt.close(fig)
(OUT/'evaluation_status.json').write_text(json.dumps(dict(status='complete',jobs=12,candidates=6,
    city_images_per_candidate=500,acdc_images_per_candidate=406,target_training=False),indent=2)+'\n')
export_workspace(OUT,ARCHIVE)
from IPython.display import Image as DisplayImage
display(DisplayImage(filename=str(OUT/'comparison.png')))
print('COMPLETE: 12 evaluations. Send dg_evaluation_seed42.zip for verification.')
display(FileLink(str(ARCHIVE)))
print('Results contain dataset panels; keep the Kaggle notebook/output private.')
